In [72]:
import numpy as np
import pandas as pd

In [73]:
path = "../data"
train_data = pd.read_csv(path + "/train.csv", encoding="latin1")
test_data = pd.read_csv(path + "/test.csv", encoding="latin1")

In [74]:
train_data.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [75]:
print(train_data["text"][10])
train_data["sentiment"][10]

 as much as i love to be hopeful, i reckon the chances are minimal =P i`m never gonna get my cake and stuff


'neutral'

In [76]:
train_data.shape

(27481, 10)

In [77]:
train_data.columns

Index(['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'],
      dtype='str')

In [78]:
train_data = pd.concat((train_data["text"],train_data["sentiment"]),axis = 1)


In [79]:
train_data.head()

,text,sentiment
0,"I`d have responded, if I were going",neutral
1,Sooo SAD I will miss you here in San Diego!!!,negative
2,my boss is bullying me...,negative
3,what interview! leave me alone,negative
4,"Sons of ****, why couldn`t they put them on t...",negative


In [80]:
train_data = train_data[(train_data["sentiment"] == "positive") | (train_data["sentiment"] == "negative")]
test_data = test_data[(test_data["sentiment"] == "positive") | (test_data["sentiment"] == "negative")]

In [81]:
train_data.shape

(16363, 2)

In [82]:
train_data["sentiment"].value_counts()

sentiment
positive    8582
negative    7781
Name: count, dtype: int64

In [83]:
train_data.isna().sum()

text         0
sentiment    0
dtype: int64

In [84]:
train_data.duplicated().sum()

np.int64(0)

In [85]:
sentiment_map = {"positive":1,
                 "negative":0}
x_train = np.array(train_data["text"])
y_train = np.array([sentiment_map[i] for i in train_data["sentiment"]])

In [86]:
x_test = np.array(test_data["text"])
y_test = np.array([sentiment_map[i] for i in test_data["sentiment"]])

In [87]:
y_train

array([0, 0, 0, ..., 0, 1, 1], shape=(16363,))

In [88]:
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re

nltk.download("stopwords")
nltk.download('punkt')

/home/yousef-magdy/miniforge3/envs/dl/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/yousef-magdy/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to /home/yousef-
[nltk_data]     magdy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/yousef-
[nltk_data]     magdy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [89]:
def preprocess(sentence):
    stemer = PorterStemmer()
    sentence = sentence.lower()
    sentence = re.sub(r"http\S+", "", sentence)
    sentence = re.sub(r"<.*?>", "", sentence)
    sentence = re.sub(r"\s+", " ", sentence).strip()
    sentence = re.sub(r'[^\w\s]', '', sentence)

    sentence = word_tokenize(sentence)
    tokens=[]
    for word in sentence:
        if word not in stopwords.words("english"):
            word = stemer.stem(word)
            tokens.append(word)
    return tokens

            
                

In [90]:
print(x_train[5])
preprocess(x_train[5])

 Journey!? Wow... u just became cooler.  hehe... (is that possible!?)


['journey', 'wow', 'u', 'becam', 'cooler', 'hehe', 'possibl']

In [91]:
freqs = {}

for sentence,y in zip(x_train,y_train):
    sentence = preprocess(sentence)
    for word in sentence:
        if word not in freqs.keys():
            word_vec = np.zeros(3)
            word_vec[0] = 1
            word_vec[2 - y] = 1 
            freqs[word] = word_vec
        else:
            freqs[word][2 - y] +=1



In [92]:
sentence_vectors = np.zeros((len(y_train), 3))
for i,sentence in enumerate(x_train):
    for word in preprocess(sentence):
        sentence_vectors[i] += freqs.get(word,0)

    sentence_vectors[i][0] = 1




In [93]:
def get_setence_vector(sentence,freqs,preprocess = preprocess):
    sentence_vector = np.zeros(3)
    for word in preprocess(sentence):
        sentence_vector += freqs.get(word,0)
    sentence_vector[0] = 1
    return sentence_vector


print(get_setence_vector("that was good",freqs))

[1.000e+00 1.056e+03 2.030e+02]


In [94]:
def sigmoid(z):

    return 1/(1 + np.exp(-z))

print(sigmoid(5))

0.9933071490757153


In [95]:
y_train= y_train.reshape(-1,1)

In [96]:
sentence_vectors

array([[1.000e+00, 1.290e+02, 1.111e+03],
       [1.000e+00, 5.000e+00, 1.300e+01],
       [1.000e+00, 7.800e+01, 1.230e+02],
       ...,
       [1.000e+00, 8.600e+02, 8.650e+02],
       [1.000e+00, 1.922e+03, 7.640e+02],
       [1.000e+00, 3.200e+01, 1.100e+01]], shape=(16363, 3))

In [ ]:
theta = np.zeros((3,1))
def gradientDescent(x, y, theta, alpha, num_iterations):
    m = len(y)
    for i in range(0, num_iterations):
        z = np.dot(x,theta)       
        h = sigmoid(z)
        J = (-1 / m) * ((np.dot(y.T, np.log(h)))  +  (np.dot(((1 - y).T), np.log(1 - h))))
        theta = theta - (alpha / m) * (np.dot(x.T , (h - y)))
    return J, theta

In [141]:
j , theta = gradientDescent(sentence_vectors,y_train,theta,0.0000001,2000)

In [142]:
j

array([[0.48199588]])

In [143]:
theta

array([[-4.67185419e-06],
       [ 2.60359411e-03],
       [-2.80632200e-03]])

In [144]:
def predict(tweet, freqs, theta):
    x = get_setence_vector(tweet,freqs)
    y_pred = sigmoid(np.dot(x , theta))
    
    return y_pred

In [145]:
def test_logistic_regression(test_x, test_y, freqs, theta, predict=predict):
    y_hat = []
    
    for tweet in test_x:
        y_pred = predict(tweet,freqs,theta)
        
        if y_pred > 0.5:
            y_hat.append(1.0)
        else:
            y_hat.append(0.0)

    accuracy = sum(np.array(y_hat) == test_y.flatten()) / len(y_hat)    
    return accuracy

In [146]:
test_logistic_regression(x_test,y_test,freqs,theta)

np.float64(0.8203422053231939)